# Hanoi & HCMC, Sentinel-2 median, GHSL — zero-shot transfer using Milan's ALL-POINTS model

Applies the Milan ALL-POINTS-trained S2-median model (no train/test split) directly to Hanoi and HCMC's own S2 median composites, with no local retraining. This is the all-points counterpart to the existing zero-shot rasters, which transfer the Milan SPLIT-trained model instead.

## Cell 1 · Imports & configuration

In [1]:
import ee
from gee_init import init_gee

init_gee()

import json
import os
import time
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload
from shapely.geometry import box

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
UTM_CRS = 'EPSG:32648'
DRIVE_FOLDER = 'GEE_exports'
CITIES = {'Hanoi': (105.85, 21.03), 'HCMC': (106.70, 10.78)}
from s2_utils import build_composite

GEE initialised (project: imss21).


---
## Cell 2 · Load Milan's all-points model + training data, build the GEE classifier

In [2]:
TARGET = 'GHSL'
LABEL_COL = 'GHSL'

with open('./output/milan/ghsl/median_alltrain/model_metadata_alltrain.json') as f:
    milan_meta = json.load(f)
band_names = milan_meta['band_names']
best_params = milan_meta['best_params']
print(f'Milan all-points model: {len(band_names)} bands, n_points_train={milan_meta["n_points_train"]}')
print(f'best_params: {best_params}')

SRC_DIR = './output/milan/ghsl/median'
train_pts = gpd.read_file(f'{SRC_DIR}/spatial_train_pts.gpkg')
test_pts  = gpd.read_file(f'{SRC_DIR}/spatial_test_pts.gpkg')
milan_pts = pd.concat([train_pts, test_pts], ignore_index=True).dropna(subset=band_names).reset_index(drop=True)
print(f'Milan training points: {len(milan_pts)} ({len(train_pts)} + {len(test_pts)})')

milan_features = [ee.Feature(None, {**{b: float(row[b]) for b in band_names}, LABEL_COL: float(row[LABEL_COL])})
                   for _, row in milan_pts.iterrows()]
milan_fc = ee.FeatureCollection(milan_features)

mf = best_params.get('max_features', 'sqrt')
if mf in ('sqrt', None):
    vps = max(1, int(np.sqrt(len(band_names))))
elif mf == 'log2':
    vps = max(1, int(np.log2(len(band_names))))
else:
    try:
        vps = max(1, int(float(mf) * len(band_names)))
    except (TypeError, ValueError):
        vps = 8

milan_rf_gee = (
    ee.Classifier.smileRandomForest(
        numberOfTrees=int(best_params.get('n_estimators', 500)),
        variablesPerSplit=vps,
        minLeafPopulation=int(best_params.get('min_samples_leaf', 1)),
        bagFraction=float(best_params.get('max_samples') or 0.8),
        seed=RANDOM_STATE)
    .setOutputMode('REGRESSION')
    .train(features=milan_fc, classProperty=LABEL_COL, inputProperties=band_names))
print('GEE classifier trained on Milan all-points data.')

Milan all-points model: 10 bands, n_points_train=3448
best_params: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_samples': 0.7, 'max_features': 'log2', 'max_depth': None}
Milan training points: 3448 (2450 + 998)


GEE classifier trained on Milan all-points data.


---
## Cell 3 · Classify each city's own composite, export

In [3]:
for CITY, (lon, lat) in CITIES.items():
    print(f'\n=== {CITY} ===')
    centre_gdf = gpd.GeoDataFrame(geometry=gpd.points_from_xy([lon], [lat]), crs='EPSG:4326').to_crs(UTM_CRS)
    cx, cy = centre_gdf.geometry.x[0], centre_gdf.geometry.y[0]
    bbox_utm  = box(cx - 15000, cy - 15000, cx + 15000, cy + 15000)
    bbox_4326 = gpd.GeoSeries([bbox_utm], crs=UTM_CRS).to_crs('EPSG:4326').iloc[0]
    aoi_geom  = ee.Geometry.Polygon([[list(c) for c in bbox_4326.exterior.coords]])

    with open(f'./output/transfer_vietnam/{CITY.lower()}/median/s2_extraction_metadata.json') as f:
        s2_meta = json.load(f)
    SELECTED_DATES = s2_meta['selected_dates']
    city_img, s2_proj, city_band_names = build_composite(
        'COPERNICUS/S2_SR_HARMONIZED', SELECTED_DATES, aoi_geom, method='median')
    assert city_band_names == band_names, f'band mismatch: {city_band_names} vs {band_names}'

    pred_img = city_img.classify(milan_rf_gee).rename('IMD_predicted').clamp(0, 100)

    OUTPUT_DIR = f'./output/transfer_vietnam/{CITY.lower()}/median_zeroshot_alltrain'
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    fname = f'{LABEL_COL}_{CITY}_10m_zeroshot_alltrain_S2median'
    out_tif = f'{OUTPUT_DIR}/{fname}.tif'
    if os.path.exists(out_tif):
        print(f'  raster already present -- skipping export.')
        continue

    task = ee.batch.Export.image.toDrive(
        image=pred_img, description=fname, folder=DRIVE_FOLDER,
        fileNamePrefix=fname, region=aoi_geom, scale=10, crs=UTM_CRS,
        maxPixels=1e10, fileFormat='GeoTIFF')
    task.start()
    print(f'  submitted batch export task (id={task.id})')
    state = 'READY'
    while state not in ('COMPLETED', 'FAILED', 'CANCELLED'):
        time.sleep(15)
        state = task.status()['state']
        print(f'    {state}')
    if state != 'COMPLETED':
        raise RuntimeError(f'export {state} -- {task.status()}')
    creds = ee.data.get_persistent_credentials()
    drive = gbuild('drive', 'v3', credentials=creds)
    resp = drive.files().list(q=f"name = '{fname}.tif' and trashed = false",
                               fields='files(id, createdTime)').execute()
    files = sorted(resp.get('files', []), key=lambda f: f['createdTime'], reverse=True)
    if not files:
        raise RuntimeError(f'export completed but {fname}.tif not found on Drive.')
    request = drive.files().get_media(fileId=files[0]['id'])
    with open(out_tif, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    print(f'  downloaded -> {out_tif}')

    meta = {'band_names': band_names, 'best_params': best_params, 'n_milan_points': len(milan_pts),
            'dates': SELECTED_DATES,
            'source': 'Milan all-points (median_alltrain) model, zero-shot transfer -- no local retrain'}
    with open(f'{OUTPUT_DIR}/model_metadata_zeroshot_alltrain.json', 'w') as f:
        json.dump(meta, f, indent=2, default=str)

print('\nDone.')


=== Hanoi ===


  submitted batch export task (id=7I5YJXENULOJSHBQ2K5SWIPZ)


    RUNNING


    RUNNING


    RUNNING


...

    RUNNING


    RUNNING


    RUNNING


    COMPLETED


  downloaded -> ./output/transfer_vietnam/hcmc/median_zeroshot_alltrain/GHSL_HCMC_10m_zeroshot_alltrain_S2median.tif

Done.
